In [1]:
!pip install torch torch-geometric networkx scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import random

from torch_geometric.utils import from_networkx
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score

In [3]:
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
def create_grid_graph(n=20, feat_dim=32):
    G = nx.grid_2d_graph(n, n)
    G = nx.convert_node_labels_to_integers(G)

    for node in G.nodes():
        G.nodes[node]['x'] = np.random.randn(feat_dim)

    data = from_networkx(G)

    data.x = torch.tensor(
        np.array([G.nodes[i]['x'] for i in range(len(G.nodes()))]),
        dtype=torch.float
    )

    return data

data = create_grid_graph()
print(data)

Data(x=[400, 32], edge_index=[2, 1520])


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/convert.py:278: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  data_dict[key] = torch.as_tensor(value)


In [5]:
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True
)

train_data, val_data, test_data = transform(data)

In [6]:
def compute_entropy(G, node):
    dist = nx.single_source_shortest_path_length(G, node)
    vals = list(dist.values())

    counts = np.bincount(vals)
    probs = counts / counts.sum()

    entropy = -sum(
        p * np.log(p + 1e-9)
        for p in probs if p > 0
    )

    return entropy

In [7]:
def build_anchor_sets(data):
    G = nx.Graph()

    edge_index = data.edge_index.cpu().numpy()

    for i in range(edge_index.shape[1]):
        G.add_edge(
            int(edge_index[0, i]),
            int(edge_index[1, i])
        )

    n = data.num_nodes

    entropy_scores = [
        compute_entropy(G, v)
        for v in range(n)
    ]

    ranked = np.argsort(entropy_scores)[::-1]

    anchors = ranked[:64]

    return anchors, G

In [8]:
anchors, G = build_anchor_sets(train_data)

print(len(anchors))

64


In [9]:
def compute_explicit_anchor_features(data, anchors, G):
    n = data.num_nodes

    feat = np.zeros((n, len(anchors)))

    for v in range(n):
        for i, a in enumerate(anchors):
            try:
                d = nx.shortest_path_length(
                    G,
                    v,
                    int(a)
                )

                feat[v, i] = 1.0 / (d + 1)

            except:
                feat[v, i] = 0.0

    return torch.tensor(feat, dtype=torch.float)

In [10]:
anchor_feat = compute_explicit_anchor_features(
    train_data,
    anchors,
    G
)

print(anchor_feat.shape)

torch.Size([400, 64])


In [11]:
class IEA_GNN_E_2L(nn.Module):
    def __init__(self, in_dim, anchor_dim, hidden=64):
        super().__init__()

        self.lin_anchor = nn.Linear(anchor_dim, hidden)

        self.lin1 = nn.Linear(
            in_dim + hidden,
            hidden
        )

        self.lin2 = nn.Linear(
            hidden,
            hidden
        )

        self.dropout = nn.Dropout(0.5)

    def aggregate(self, x, edge_index):
        row, col = edge_index

        out = torch.zeros_like(x)
        out.index_add_(0, row, x[col])

        deg = torch.bincount(
            row,
            minlength=x.size(0)
        ).float().unsqueeze(1)

        deg[deg == 0] = 1

        return out / deg

    def forward(self, x, anchor_feat, edge_index):
        pos = F.relu(self.lin_anchor(anchor_feat))

        h = torch.cat([x, pos], dim=1)

        h = self.aggregate(h, edge_index)
        h = F.relu(self.lin1(h))
        h = self.dropout(h)

        h = self.aggregate(h, edge_index)
        h = self.lin2(h)

        return h

In [12]:
model = IEA_GNN_E_2L(
    train_data.x.shape[1],
    anchor_feat.shape[1],
    hidden=64
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

train_data = train_data.to(device)
test_data = test_data.to(device)
anchor_feat = anchor_feat.to(device)

In [13]:
def decode(z, edge_index):
    return (
        z[edge_index[0]] *
        z[edge_index[1]]
    ).sum(dim=1)

In [14]:
for epoch in range(2500):
    model.train()
    optimizer.zero_grad()

    z = model(
        train_data.x,
        anchor_feat,
        train_data.edge_index
    )

    pred = decode(
        z,
        train_data.edge_label_index
    )

    loss = F.binary_cross_entropy_with_logits(
        pred,
        train_data.edge_label.float()
    )

    loss.backward()
    optimizer.step()

    if epoch % 250 == 0:
        print(epoch, loss.item())

0 0.7364690899848938
250 0.07340656965970993
500 0.06238728016614914
750 0.051965806633234024
1000 0.04921461269259453
1250 0.05652691051363945
1500 0.05590987205505371
1750 0.056750133633613586
2000 0.07062482088804245
2250 0.06396330147981644


In [15]:
test_anchor_feat = compute_explicit_anchor_features(
    test_data,
    anchors,
    G
).to(device)

model.eval()

with torch.no_grad():
    z = model(
        test_data.x,
        test_anchor_feat,
        test_data.edge_index
    )

pred = torch.sigmoid(
    decode(z, test_data.edge_label_index)
).cpu().numpy()

labels = test_data.edge_label.cpu().numpy()

auc = roc_auc_score(labels, pred)

print("IEA-GNN-E-2L AUC:", auc)

IEA-GNN-E-2L AUC: 0.7435076177285319
